In [1]:
import os
import re
import html
import requests
import feedparser
import pandas as pd
import numpy as np
import io
from datetime import datetime

from datetime import datetime
from dotenv import load_dotenv
import mysql.connector

In [2]:
nse_session = requests.Session()

nse_headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/139.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/companies-listing/corporate-filings-announcements"
}

nse_session.headers.update(nse_headers)

# First visit NSE page to establish session/cookies
page_response = nse_session.get(
    "https://www.nseindia.com/companies-listing/corporate-filings-announcements",
    timeout=30
)

print("NSE page status:", page_response.status_code)

NSE page status: 200


In [3]:
from_date = "01-08-2026"
to_date   = "31-08-2026"

api_url = "https://www.nseindia.com/api/corporate-announcements"

params = {
    "index": "equities",
    "from_date": from_date,
    "to_date": to_date
}

response = nse_session.get(
    api_url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("Content length:", len(response.content))
print("Content-Type:", response.headers.get("Content-Type"))

Status: 200
Content length: 14644258
Content-Type: application/json; charset=utf-8


In [4]:
print(response.text[:500])

[{"an_dt":"31-Aug-2026 23:48:18","attFileSize":"344.87 KB","attchmntFile":"https://nsearchives.nseindia.com/corporate/MUTHOOTFIN_31082026234806_AGM_SD.pdf","attchmntText":"Muthoot Finance Limited has informed the Exchange regarding Proceedings of Annual General Meeting held on August 31, 2026","bflag":null,"csvName":null,"desc":"Shareholders meeting","difference":"00:00:01","dt":"31082026234818","exchdisstime":"31-Aug-2026 23:48:19","fileSize":"344.87 KB","hasXbrl":true,"old_new":null,"orgid":nu


In [5]:
data = response.json()

print(type(data))

if isinstance(data, list):
    print("Records:", len(data))
    display(pd.DataFrame(data).head())

elif isinstance(data, dict):
    print("Keys:", data.keys())

<class 'list'>
Records: 21024


,an_dt,attFileSize,attchmntFile,attchmntText,bflag,csvName,desc,difference,dt,exchdisstime,fileSize,hasXbrl,old_new,orgid,seq_id,smIndustry,sm_isin,sm_name,sort_date,symbol
0,31-Aug-2026 23:48:18,344.87 KB,https://nsearchives.nseindia.com/corporate/MUT...,Muthoot Finance Limited has informed the Excha...,None,None,Shareholders meeting,00:00:01,31082026234818,31-Aug-2026 23:48:19,344.87 KB,True,None,None,106763396,Finance,INE414G01012,Muthoot Finance Limited,2026-08-31 23:48:18,MUTHOOTFIN
1,31-Aug-2026 23:36:42,557.77 KB,https://nsearchives.nseindia.com/corporate/GIC...,General Insurance Corporation of India has inf...,None,None,Shareholders meeting,00:00:01,31082026233642,31-Aug-2026 23:36:43,557.77 KB,True,None,None,106763385,None,INE481Y01014,General Insurance Corporation of India,2026-08-31 23:36:42,GICRE
2,31-Aug-2026 23:36:14,2.01 MB,https://nsearchives.nseindia.com/corporate/Dim...,Just Dial Limited has informed the Exchange re...,None,None,Change in Director(s),00:00:01,31082026233614,31-Aug-2026 23:36:15,2.01 MB,True,None,None,106763384,None,INE599M01018,Just Dial Limited,2026-08-31 23:36:14,JUSTDIAL
3,31-Aug-2026 23:31:38,2.01 MB,https://nsearchives.nseindia.com/corporate/chi...,Just Dial Limited has submitted the Exchange a...,None,None,Shareholders meeting,00:00:01,31082026233138,31-Aug-2026 23:31:39,2.01 MB,True,None,None,106763382,None,INE599M01018,Just Dial Limited,2026-08-31 23:31:38,JUSTDIAL
4,31-Aug-2026 23:27:56,298.88 KB,https://nsearchives.nseindia.com/corporate/E2E...,E2E Networks Limited has informed the Exchange...,None,None,Press Release,00:00:01,31082026232756,31-Aug-2026 23:27:57,298.88 KB,True,None,None,106763381,None,INE255Z01019,E2E Networks Limited,2026-08-31 23:27:56,E2E


In [6]:
# Convert NSE response to DataFrame
nse_df = pd.DataFrame(data)

print("Rows:", len(nse_df))
print("\nColumns:")
for col in nse_df.columns:
    print("-", col)

print("\nMissing values:")
display(nse_df.isna().sum().sort_values(ascending=False))

print("\nUnique companies:")
print("Symbols:", nse_df["symbol"].nunique())
print("ISINs:", nse_df["sm_isin"].nunique())
print("Company names:", nse_df["sm_name"].nunique())

print("\nDate range:")
print("Oldest:", nse_df["an_dt"].min())
print("Latest:", nse_df["an_dt"].max())

print("\nAnnouncement types:")

Rows: 21024

Columns:
- an_dt
- attFileSize
- attchmntFile
- attchmntText
- bflag
- csvName
- desc
- difference
- dt
- exchdisstime
- fileSize
- hasXbrl
- old_new
- orgid
- seq_id
- smIndustry
- sm_isin
- sm_name
- sort_date
- symbol

Missing values:


bflag           21024
csvName         21024
orgid           21024
old_new         21024
smIndustry      12981
an_dt               0
hasXbrl             0
sort_date           0
sm_name             0
sm_isin             0
seq_id              0
fileSize            0
attFileSize         0
exchdisstime        0
dt                  0
difference          0
desc                0
attchmntText        0
attchmntFile        0
symbol              0
dtype: int64


Unique companies:
Symbols: 2329
ISINs: 2328
Company names: 2326

Date range:
Oldest: 01-Aug-2026 08:11:24
Latest: 31-Aug-2026 23:48:18

Announcement types:


In [7]:
print("\nDate range:")
print("Oldest:", nse_df["an_dt"].min())
print("Latest:", nse_df["an_dt"].max())

print("\nAnnouncement types:")
display(
    nse_df["desc"]
    .value_counts()
    .head(30)
)


Date range:
Oldest: 01-Aug-2026 08:11:24
Latest: 31-Aug-2026 23:48:18

Announcement types:


desc
Analysts/Institutional Investor Meet/Con. Call Updates                       3442
Copy of Newspaper Publication                                                3083
General Updates                                                              2210
Outcome of Board Meeting                                                     2188
Shareholders meeting                                                         2015
Updates                                                                      1418
Press Release                                                                 966
Investor Presentation                                                         724
Record Date                                                                   502
Appointment                                                                   502
Change in Management                                                          346
Monitoring Agency Report                                                      274
Resignation

In [11]:
news_df = nse_df.copy()

news_df = news_df.rename(columns={
    "an_dt": "published_at",
    "sm_isin": "isin",
    "sm_name": "company_name",
    "desc": "event_type",
    "attchmntFile": "attachment_url",
    "attchmntText": "attachment_text",
    "seq_id": "nse_seq_id",
    "smIndustry": "industry",
    "sort_date": "sort_date",
})

news_df = news_df[
    [
        "nse_seq_id",
        "symbol",
        "isin",
        "company_name",
        "event_type",
        "published_at",
        "sort_date",
        "attachment_url",
        "attachment_text",
        "industry",
        "hasXbrl"
    ]
]

print("Rows:", len(news_df))
display(news_df.head())

Rows: 21024


,nse_seq_id,symbol,isin,company_name,event_type,published_at,sort_date,attachment_url,attachment_text,industry,hasXbrl
0,106763396,MUTHOOTFIN,INE414G01012,Muthoot Finance Limited,Shareholders meeting,31-Aug-2026 23:48:18,2026-08-31 23:48:18,https://nsearchives.nseindia.com/corporate/MUT...,Muthoot Finance Limited has informed the Excha...,Finance,True
1,106763385,GICRE,INE481Y01014,General Insurance Corporation of India,Shareholders meeting,31-Aug-2026 23:36:42,2026-08-31 23:36:42,https://nsearchives.nseindia.com/corporate/GIC...,General Insurance Corporation of India has inf...,None,True
2,106763384,JUSTDIAL,INE599M01018,Just Dial Limited,Change in Director(s),31-Aug-2026 23:36:14,2026-08-31 23:36:14,https://nsearchives.nseindia.com/corporate/Dim...,Just Dial Limited has informed the Exchange re...,None,True
3,106763382,JUSTDIAL,INE599M01018,Just Dial Limited,Shareholders meeting,31-Aug-2026 23:31:38,2026-08-31 23:31:38,https://nsearchives.nseindia.com/corporate/chi...,Just Dial Limited has submitted the Exchange a...,None,True
4,106763381,E2E,INE255Z01019,E2E Networks Limited,Press Release,31-Aug-2026 23:27:56,2026-08-31 23:27:56,https://nsearchives.nseindia.com/corporate/E2E...,E2E Networks Limited has informed the Exchange...,None,True


In [12]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv("/Users/amit/Desktop/InvestIQ/backend/.env")

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor(dictionary=True)

cursor.execute("""
    SELECT
        id,
        symbol,
        name,
        isin
    FROM companies
    WHERE isin IS NOT NULL
""")

companies_df = pd.DataFrame(cursor.fetchall())

cursor.close()
conn.close()

print("Companies in DB:", len(companies_df))
display(companies_df.head())

Companies in DB: 3117


,id,symbol,name,isin
0,5,20MICRONS,20 Microns Limited,INE144J01027
1,6,21STCENMGM,21st Century Management Services Limited,INE253B01015
2,7,360ONE,360 ONE WAM LIMITED,INE466L01038
3,8,3BBLACKBIO,3B Blackbio Dx Limited,INE994E01018
4,9,3IINFOLTD,3i Infotech Limited,INE748C01038


In [13]:
news_df = news_df.merge(
    companies_df[["id", "symbol", "isin"]],
    on="isin",
    how="left",
    suffixes=("", "_db")
)

news_df = news_df.rename(columns={
    "id": "company_id"
})

print("Total NSE announcements:", len(news_df))

print(
    "Mapped to InvestIQ:",
    news_df["company_id"].notna().sum()
)

print(
    "Unmapped:",
    news_df["company_id"].isna().sum()
)

print(
    "Mapping percentage:",
    round(
        news_df["company_id"].notna().mean() * 100,
        2
    ),
    "%"
)

Total NSE announcements: 21024
Mapped to InvestIQ: 16448
Unmapped: 4576
Mapping percentage: 78.23 %


In [14]:
unmapped = (
    news_df[news_df["company_id"].isna()]
    [["symbol", "isin", "company_name"]]
    .drop_duplicates()
)

print("Unique unmapped companies:", len(unmapped))

display(unmapped.head(50))

Unique unmapped companies: 523


,symbol,isin,company_name
4,E2E,INE255Z01019,E2E Networks Limited
11,BRITANNIA,INE216A01014,Britannia Industries Limited
14,PERSISTENT,INE262H01013,Persistent Systems Limited
16,LICHSGFIN,INE115A01018,LIC Housing Finance Limited
20,TATASTEEL,INE081A01012,Tata Steel Limited
21,NAZARA,INE418L01021,Nazara Technologies Limited
26,AXISBANK,INE238A01026,Axis Bank Limited
35,PUNJLLOYD,INE701B01021,Punj Lloyd Limited
37,FILATEX,INE816B01019,Filatex India Limited
45,BANKBARODA,INE028A01013,Bank of Baroda


In [15]:
# Check whether the supposedly unmapped ISINs
# actually exist in our companies table

unmapped = news_df[news_df["company_id"].isna()].copy()

# Normalize ISINs
unmapped["isin_clean"] = (
    unmapped["isin"]
    .astype(str)
    .str.strip()
    .str.upper()
)

companies_df["isin_clean"] = (
    companies_df["isin"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Check direct ISIN match
isin_exists = unmapped["isin_clean"].isin(
    companies_df["isin_clean"]
)

print("Unmapped rows:", len(unmapped))
print("Unmapped rows whose ISIN exists in DB:", isin_exists.sum())
print(
    "Percentage:",
    round(isin_exists.mean() * 100, 2),
    "%"
)

Unmapped rows: 4576
Unmapped rows whose ISIN exists in DB: 0
Percentage: 0.0 %


In [16]:
debug_match = unmapped[
    isin_exists
][
    ["symbol", "isin", "company_name"]
].drop_duplicates()

print("Companies that SHOULD have matched:", len(debug_match))

display(debug_match.head(50))

Companies that SHOULD have matched: 0


,symbol,isin,company_name


In [19]:
# Inspect unmapped NSE announcements directly from the original NSE dataframe

unmapped_symbols = (
    news_df[news_df["company_id"].isna()]
    ["symbol"]
    .drop_duplicates()
)

print("Unique unmapped symbols:", len(unmapped_symbols))

display(
    nse_df[
        nse_df["symbol"].isin(unmapped_symbols)
    ][
        ["symbol", "sm_isin", "sm_name", "smIndustry"]
    ]
    .drop_duplicates()
    .head(100)
)

Unique unmapped symbols: 523


,symbol,sm_isin,sm_name,smIndustry
4,E2E,INE255Z01019,E2E Networks Limited,None
11,BRITANNIA,INE216A01014,Britannia Industries Limited,Food And Food Processing
14,PERSISTENT,INE262H01013,Persistent Systems Limited,Computers - Software
16,LICHSGFIN,INE115A01018,LIC Housing Finance Limited,Finance - Housing
20,TATASTEEL,INE081A01012,Tata Steel Limited,Steel And Steel Products
...,...,...,...,...
601,JTLIND,INE391J01024,JTL INDUSTRIES LIMITED,None
615,TALBROAUTO,INE187D01011,Talbros Automotive Components Limited,Auto Ancillaries
616,BALAXI,INE618N01014,BALAXI PHARMACEUTICALS LIMITED,None
618,GPIL,INE177H01013,Godawari Power And Ispat limited,Steel And Steel Products


In [20]:
# Compare NSE symbols against InvestIQ symbols

nse_symbols = set(
    nse_df["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

db_symbols = set(
    companies_df["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

common_symbols = nse_symbols & db_symbols
only_nse_symbols = nse_symbols - db_symbols

print("NSE unique symbols:", len(nse_symbols))
print("DB unique symbols:", len(db_symbols))
print("Common symbols:", len(common_symbols))
print("Only NSE symbols:", len(only_nse_symbols))

print("\nOnly NSE examples:")
print(list(sorted(only_nse_symbols))[:100])

NSE unique symbols: 2329
DB unique symbols: 3117
Common symbols: 2264
Only NSE symbols: 65

Only NSE examples:
['ABHISHEK', 'AIFL', 'AJRINFRA', 'ALPSINDUS', 'ARSSINFRA', 'ASIL', 'ATCOM', 'ATNINTER', 'AUGMONT', 'BALLARPUR', 'BGLOBAL', 'BHARATIDIL', 'BILENERGY', 'BKMINDST', 'BLUEBLENDS', 'CANDC', 'CMICABLES', 'CRESTO', 'DHARSUGAR', 'DIL', 'ERAINFRA', 'EROSMEDIA', 'FCONSUMER', 'GAJA', 'GAMMONIND', 'GANGOTRI', 'GBGLOBAL', 'GFSTEELS', 'GOENKA', 'HORIZONIND', 'ICSA', 'INDLMETER', 'INDUSFILA', 'JBFIND', 'KDGREEN', 'KSERASERA', 'KSOILS', 'KUNDANMM', 'LALITHAA', 'LEEL', 'MBECL', 'MELSTAR', 'NAGAFERT', 'NITINFIRE', 'OMKARCHEM', 'PODDARHOUS', 'PUNJLLOYD', 'QUINTEGRA', 'RMCL', 'RUSHABEAR', 'SABEVENTS', 'SECURCRED', 'SHANKESH', 'SKIL', 'SRPL', 'SUNSHINE', 'TALWALKARS', 'TECHIN', 'TEMPSENS', 'UNIVAFOODS', 'VALECHAENG', 'VALUEIND', 'VIDEOIND', 'WINSOME', 'XLENERGY']


In [21]:
# Create a clean list of the 523 unmapped symbols

unmapped_symbols_df = (
    news_df[news_df["company_id"].isna()]
    [
        [
            "symbol",
            "isin",
            "company_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("symbol")
)

print("Unique unmapped symbols:", len(unmapped_symbols_df))

display(unmapped_symbols_df.head(100))

Unique unmapped symbols: 523


,symbol,isin,company_name
8630,20MICRONS,INE144J01019,20 Microns Limited
1184,360ONE,INE466L01020,360 ONE WAM LIMITED
401,AAKASH,INE087Z01016,Aakash Exploration Services Limited
7153,AARTECH,INE01C001018,Aartech Solonics Limited
7149,ABHISHEK,INE004I01017,Abhishek Corporation Limited
...,...,...,...
8939,BTML,INE0EEJ01015,Bodhi Tree Multimedia Limited
980,BURNPUR,INE817H01014,Burnpur Cement Limited
534,CAMS,INE596I01012,Computer Age Management Services Limited
819,CANBK,INE476A01014,Canara Bank


In [22]:
from difflib import get_close_matches

db_symbol_list = companies_df["symbol"].dropna().astype(str).str.upper().tolist()

matches = []

for symbol in unmapped_symbols_df["symbol"].astype(str).str.upper():
    close = get_close_matches(
        symbol,
        db_symbol_list,
        n=3,
        cutoff=0.75
    )

    matches.append({
        "nse_symbol": symbol,
        "possible_db_symbols": ", ".join(close)
    })

symbol_matches_df = pd.DataFrame(matches)

display(symbol_matches_df.head(100))


,nse_symbol,possible_db_symbols
0,20MICRONS,20MICRONS
1,360ONE,360ONE
2,AAKASH,"AAKASH, AKASH, SAAKSHI"
3,AARTECH,"AARTECH, AAATECH, TATATECH"
4,ABHISHEK,
...,...,...
95,BTML,"BTML, TTML, ETML"
96,BURNPUR,BURNPUR
97,CAMS,"CAMS, CAMPUS, ASMS"
98,CANBK,CANBK


In [23]:
db_names = companies_df[
    ["id", "symbol", "name", "isin"]
].dropna(subset=["name"]).copy()

db_name_list = db_names["name"].astype(str).tolist()

name_matches = []

for _, row in unmapped_symbols_df.iterrows():

    close = get_close_matches(
        str(row["company_name"]),
        db_name_list,
        n=3,
        cutoff=0.75
    )

    name_matches.append({
        "nse_symbol": row["symbol"],
        "nse_name": row["company_name"],
        "possible_db_names": " | ".join(close)
    })

name_matches_df = pd.DataFrame(name_matches)

display(name_matches_df.head(100))

,nse_symbol,nse_name,possible_db_names
0,20MICRONS,20 Microns Limited,20 Microns Limited | Marsons Limited | Nirlon ...
1,360ONE,360 ONE WAM LIMITED,360 ONE WAM LIMITED | SONAM LIMITED
2,AAKASH,Aakash Exploration Services Limited,Aakash Exploration Services Limited
3,AARTECH,Aartech Solonics Limited,Aartech Solonics Limited
4,ABHISHEK,Abhishek Corporation Limited,Hitech Corporation Limited | Abhishek Integrat...
...,...,...,...
95,BTML,Bodhi Tree Multimedia Limited,Bodhi Tree Multimedia Limited
96,BURNPUR,Burnpur Cement Limited,Burnpur Cement Limited | Orient Cement Limited...
97,CAMS,Computer Age Management Services Limited,Computer Age Management Services Limited | Rap...
98,CANBK,Canara Bank,Canara Bank


In [24]:
# Check whether each DB symbol identifies only one company

symbol_counts = (
    companies_df
    .groupby("symbol")
    .size()
    .sort_values(ascending=False)
)

print("Total DB symbols:", len(symbol_counts))
print("Symbols mapping to >1 company:", (symbol_counts > 1).sum())

display(
    symbol_counts[symbol_counts > 1].head(30)
)

Total DB symbols: 3117
Symbols mapping to >1 company: 0


Series([], dtype: int64)

In [25]:
# Create clean DB symbol lookup

company_symbol_lookup = (
    companies_df[
        ["id", "symbol", "name", "isin"]
    ]
    .dropna(subset=["symbol"])
    .copy()
)

company_symbol_lookup["symbol"] = (
    company_symbol_lookup["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Only use symbols that uniquely identify one company
unique_symbol_lookup = (
    company_symbol_lookup
    .groupby("symbol")
    .filter(lambda x: len(x) == 1)
)

print(
    "Unique DB symbols available for mapping:",
    unique_symbol_lookup["symbol"].nunique()
)

Unique DB symbols available for mapping: 3117


In [26]:
# Normalize NSE symbol

news_df["symbol_clean"] = (
    news_df["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

symbol_map = (
    unique_symbol_lookup
    .set_index("symbol")["id"]
    .to_dict()
)

news_df["company_id_symbol"] = (
    news_df["symbol_clean"]
    .map(symbol_map)
)

print("Total announcements:", len(news_df))

print(
    "Mapped by symbol:",
    news_df["company_id_symbol"].notna().sum()
)

print(
    "Still unmapped:",
    news_df["company_id_symbol"].isna().sum()
)

print(
    "Mapping percentage:",
    round(
        news_df["company_id_symbol"].notna().mean() * 100,
        2
    ),
    "%"
)

Total announcements: 21024
Mapped by symbol: 20773
Still unmapped: 251
Mapping percentage: 98.81 %


In [27]:
print("Original ISIN mapping:")
print(
    news_df["company_id"].notna().sum()
)

print("\nSymbol mapping:")
print(
    news_df["company_id_symbol"].notna().sum()
)

Original ISIN mapping:
16448

Symbol mapping:
20773


In [28]:
symbol_only = news_df[
    news_df["company_id"].isna()
    &
    news_df["company_id_symbol"].notna()
].copy()

print(
    "Recovered using symbol:",
    len(symbol_only)
)

display(
    symbol_only[
        [
            "symbol",
            "isin",
            "company_name",
            "company_id_symbol"
        ]
    ]
    .drop_duplicates()
    .head(50)
)

Recovered using symbol: 4327


,symbol,isin,company_name,company_id_symbol
4,E2E,INE255Z01019,E2E Networks Limited,640.0
11,BRITANNIA,INE216A01014,Britannia Industries Limited,401.0
14,PERSISTENT,INE262H01013,Persistent Systems Limited,1691.0
16,LICHSGFIN,INE115A01018,LIC Housing Finance Limited,1308.0
20,TATASTEEL,INE081A01012,Tata Steel Limited,2251.0
21,NAZARA,INE418L01021,Nazara Technologies Limited,1526.0
26,AXISBANK,INE238A01026,Axis Bank Limited,263.0
37,FILATEX,INE816B01019,Filatex India Limited,734.0
45,BANKBARODA,INE028A01013,Bank of Baroda,302.0
47,DIXON,INE935N01012,Dixon Technologies (India) Limited,609.0


In [29]:
news_df["final_company_id"] = (
    news_df["company_id"]
    .fillna(news_df["company_id_symbol"])
)

In [30]:
still_unmapped = news_df[
    news_df["company_id_symbol"].isna()
].copy()

print("Unmapped announcements:", len(still_unmapped))
print(
    "Unique unmapped symbols:",
    still_unmapped["symbol"].nunique()
)

display(
    still_unmapped[
        [
            "symbol",
            "isin",
            "company_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("symbol")
)

Unmapped announcements: 251
Unique unmapped symbols: 65


,symbol,isin,company_name
7149,ABHISHEK,INE004I01017,Abhishek Corporation Limited
7042,AIFL,INE428O01016,Ashapura Intimates Fashion Limited
726,AJRINFRA,INE181G01025,AJR INFRA AND TOLLING LIMITED
2857,ALPSINDUS,INE093B01015,Alps Industries Limited
497,ARSSINFRA,INE267I01010,ARSS Infrastructure Projects Limited
...,...,...,...
1044,VALECHAENG,INE624C01015,Valecha Engineering Limited
6367,VALUEIND,INE352A01017,Value Industries Limited
6365,VIDEOIND,INE703A01011,Videocon Industries Limited
451,WINSOME,INE784B01035,Winsome Yarns Limited


In [31]:
print("Unmapped announcement types:")

display(
    still_unmapped["event_type"]
    .value_counts()
    .head(30)
)

Unmapped announcement types:


event_type
Outcome of Board Meeting                                   60
Copy of Newspaper Publication                              40
General Updates                                            25
Updates                                                    24
Corporate Insolvency Resolution Process                    14
Shareholders meeting                                       13
Change in Management                                       10
Trading Window                                              7
Appointment                                                 5
Resignation of Director/KMP/SMP                             5
Integrated Filing- Financial                                5
Resignation                                                 4
Record Date                                                 4
Change in Auditors                                          4
Reasons for Delayed/Non-submission of Financial Results     4
Cessation                                                  

In [32]:
print("Unmapped announcements by symbol:")

display(
    still_unmapped["symbol"]
    .value_counts()
    .head(50)
)

Unmapped announcements by symbol:


symbol
VALECHAENG    11
KUNDANMM      10
TALWALKARS    10
GAMMONIND      9
MELSTAR        8
XLENERGY       8
SRPL           7
QUINTEGRA      7
ARSSINFRA      7
MBECL          7
SECURCRED      6
OMKARCHEM      6
SHANKESH       5
FCONSUMER      5
SKIL           5
JBFIND         5
BLUEBLENDS     5
GANGOTRI       5
AUGMONT        5
BALLARPUR      5
AJRINFRA       5
ABHISHEK       5
KSERASERA      5
WINSOME        5
KSOILS         4
ATCOM          4
LEEL           4
GBGLOBAL       4
PUNJLLOYD      4
HORIZONIND     3
INDLMETER      3
ATNINTER       3
NITINFIRE      3
RMCL           3
DHARSUGAR      3
SABEVENTS      3
NAGAFERT       3
CANDC          3
BGLOBAL        3
GAJA           3
ALPSINDUS      3
LALITHAA       3
ERAINFRA       3
RUSHABEAR      2
INDUSFILA      2
GOENKA         2
CMICABLES      2
UNIVAFOODS     2
VALUEIND       2
VIDEOIND       2
Name: count, dtype: int64

In [38]:
news_df["final_company_id"] = (
    news_df["company_id"]
    .fillna(news_df["company_id_symbol"])
)

print(
    "Final mapped:",
    news_df["final_company_id"].notna().sum()
)

print(
    "Final unmapped:",
    news_df["final_company_id"].isna().sum()
)

Final mapped: 20775
Final unmapped: 249


In [33]:
cursor = conn.cursor(dictionary=True)

cursor.execute("DESCRIBE news")

news_schema = pd.DataFrame(cursor.fetchall())

display(news_schema)

cursor.close()
conn.close()

OperationalError: MySQL Connection not available.

In [34]:
import mysql.connector
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv("/Users/amit/Desktop/InvestIQ/backend/.env")

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=True
)

print("MySQL connected:", conn.is_connected())

MySQL connected: True


In [35]:
cursor = conn.cursor(dictionary=True)

cursor.execute("DESCRIBE news")

news_schema = pd.DataFrame(cursor.fetchall())

display(news_schema)

cursor.close()

,Field,Type,Null,Key,Default,Extra
0,id,bigint,NO,PRI,None,auto_increment
1,company_id,int,NO,MUL,None,
2,symbol,varchar(50),NO,,None,
3,isin,varchar(20),NO,,None,
4,company_name,varchar(255),NO,,None,
5,headline,text,NO,,None,
6,content,longtext,YES,,None,
7,event_type,varchar(150),YES,,None,
8,importance,"enum('HIGH','MEDIUM','LOW')",YES,,None,
9,feed_type,varchar(100),NO,,None,


True

In [36]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=True
)

cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
        ADD COLUMN content LONGTEXT NULL AFTER headline,
        ADD COLUMN event_type VARCHAR(150) NULL AFTER content,
        ADD COLUMN nse_seq_id BIGINT NULL AFTER source,
        ADD COLUMN attachment_url TEXT NULL AFTER nse_seq_id,
        ADD COLUMN has_xbrl BOOLEAN NULL AFTER attachment_url
""")

print("News table updated successfully.")

cursor.close()
conn.close()

ProgrammingError: 1060 (42S21): Duplicate column name 'content'

In [37]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor(dictionary=True)

cursor.execute("DESCRIBE news")

display(pd.DataFrame(cursor.fetchall()))

cursor.close()
conn.close()

,Field,Type,Null,Key,Default,Extra
0,id,bigint,NO,PRI,None,auto_increment
1,company_id,int,NO,MUL,None,
2,symbol,varchar(50),NO,,None,
3,isin,varchar(20),NO,,None,
4,company_name,varchar(255),NO,,None,
5,headline,text,NO,,None,
6,content,longtext,YES,,None,
7,event_type,varchar(150),YES,,None,
8,importance,"enum('HIGH','MEDIUM','LOW')",YES,,None,
9,feed_type,varchar(100),NO,,None,


In [38]:
cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
    ADD UNIQUE KEY unique_nse_seq (
        source,
        nse_seq_id
    )
""")

print("Unique NSE constraint added.")

cursor.close()

OperationalError: MySQL Connection not available.

In [39]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv("/Users/amit/Desktop/InvestIQ/backend/.env")

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=True
)

cursor = conn.cursor()

print("Connected:", conn.is_connected())

# Check whether the unique key already exists
cursor.execute("""
    SHOW INDEX FROM news
    WHERE Key_name = 'unique_nse_seq'
""")

existing_index = cursor.fetchall()

if existing_index:
    print("unique_nse_seq already exists.")
else:
    cursor.execute("""
        ALTER TABLE news
        ADD UNIQUE KEY unique_nse_seq (source, nse_seq_id)
    """)
    print("unique_nse_seq added successfully.")

cursor.close()
conn.close()

Connected: True
unique_nse_seq already exists.


In [46]:
display(
    investiq_news[
        [
            "symbol",
            "company_name",
            "event_type",
            "attachment_text"
        ]
    ].head(20).to_string(index=False)
)

NameError: name 'investiq_news' is not defined

In [40]:
# Recreate the final InvestIQ news dataset

news_df["final_company_id"] = (
    news_df["company_id"]
    .fillna(news_df["company_id_symbol"])
)

investiq_news = news_df[
    news_df["final_company_id"].notna()
].copy()

investiq_news["final_company_id"] = (
    investiq_news["final_company_id"]
    .astype(int)
)

print("Total NSE announcements:", len(news_df))
print("InvestIQ news records:", len(investiq_news))
print(
    "Excluded unmapped records:",
    news_df["final_company_id"].isna().sum()
)


Total NSE announcements: 21024
InvestIQ news records: 20775
Excluded unmapped records: 249


In [41]:
display(
    investiq_news[
        [
            "symbol",
            "company_name",
            "event_type",
            "attachment_text"
        ]
    ].head(20).to_string(index=False)
)

'    symbol                           company_name                      event_type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              attachment_text\nMUTHOOTFIN                Muthoot Finance Limited            Shareholders meeting                      

In [42]:
investiq_news["text_length"] = (
    investiq_news["attachment_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

display(investiq_news["text_length"].describe())

count    20775.000000
mean       133.343442
std         90.059093
min         13.000000
25%         83.000000
50%        112.000000
75%        157.000000
max       2938.000000
Name: text_length, dtype: float64

In [43]:
# Prepare final NSE dataset for MySQL ingestion

nse_insert_df = investiq_news.copy()

# Use the final mapped company ID
nse_insert_df["company_id"] = (
    nse_insert_df["final_company_id"]
    .astype(int)
)

# NSE announcement text
nse_insert_df["content"] = (
    nse_insert_df["attachment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# For now, use the NSE event description as headline.
# We can later generate better headlines with an NLP layer.
nse_insert_df["headline"] = (
    nse_insert_df["event_type"]
    .fillna("NSE Announcement")
    .astype(str)
    .str.strip()
)

# Source
nse_insert_df["source"] = "NSE"

# Feed type tells us this came from the historical API
nse_insert_df["feed_type"] = "historical_announcement"

# Event date
nse_insert_df["event_date"] = pd.to_datetime(
    nse_insert_df["published_at"],
    format="%d-%b-%Y %H:%M:%S",
    errors="coerce"
).dt.date

# Published timestamp
nse_insert_df["published_at"] = pd.to_datetime(
    nse_insert_df["published_at"],
    format="%d-%b-%Y %H:%M:%S",
    errors="coerce"
)

# URL: use attachment URL
nse_insert_df["url"] = (
    nse_insert_df["attachment_url"]
    .fillna("")
    .astype(str)
)

# Keep only the DB fields
nse_insert_df = nse_insert_df[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "content",
        "event_type",
        "feed_type",
        "published_at",
        "event_date",
        "url",
        "source",
        "nse_seq_id",
        "attachment_url",
        "hasXbrl"
    ]
].copy()

print("Records ready:", len(nse_insert_df))

display(nse_insert_df.head(10))

Records ready: 20775


,company_id,symbol,isin,company_name,headline,content,event_type,feed_type,published_at,event_date,url,source,nse_seq_id,attachment_url,hasXbrl
0,1500,MUTHOOTFIN,INE414G01012,Muthoot Finance Limited,Shareholders meeting,Muthoot Finance Limited has informed the Excha...,Shareholders meeting,historical_announcement,2026-08-31 23:48:18,2026-08-31,https://nsearchives.nseindia.com/corporate/MUT...,NSE,106763396,https://nsearchives.nseindia.com/corporate/MUT...,True
1,798,GICRE,INE481Y01014,General Insurance Corporation of India,Shareholders meeting,General Insurance Corporation of India has inf...,Shareholders meeting,historical_announcement,2026-08-31 23:36:42,2026-08-31,https://nsearchives.nseindia.com/corporate/GIC...,NSE,106763385,https://nsearchives.nseindia.com/corporate/GIC...,True
2,1158,JUSTDIAL,INE599M01018,Just Dial Limited,Change in Director(s),Just Dial Limited has informed the Exchange re...,Change in Director(s),historical_announcement,2026-08-31 23:36:14,2026-08-31,https://nsearchives.nseindia.com/corporate/Dim...,NSE,106763384,https://nsearchives.nseindia.com/corporate/Dim...,True
3,1158,JUSTDIAL,INE599M01018,Just Dial Limited,Shareholders meeting,Just Dial Limited has submitted the Exchange a...,Shareholders meeting,historical_announcement,2026-08-31 23:31:38,2026-08-31,https://nsearchives.nseindia.com/corporate/chi...,NSE,106763382,https://nsearchives.nseindia.com/corporate/chi...,True
4,640,E2E,INE255Z01019,E2E Networks Limited,Press Release,E2E Networks Limited has informed the Exchange...,Press Release,historical_announcement,2026-08-31 23:27:56,2026-08-31,https://nsearchives.nseindia.com/corporate/E2E...,NSE,106763381,https://nsearchives.nseindia.com/corporate/E2E...,True
5,2294,TI,INE133E01013,Tilaknagar Industries Limited,Record Date,Tilaknagar Industries Limited has informed the...,Record Date,historical_announcement,2026-08-31 23:26:24,2026-08-31,https://nsearchives.nseindia.com/corporate/gth...,NSE,106763379,https://nsearchives.nseindia.com/corporate/gth...,True
6,798,GICRE,INE481Y01014,General Insurance Corporation of India,Copy of Newspaper Publication,General Insurance Corporation of India has inf...,Copy of Newspaper Publication,historical_announcement,2026-08-31 23:24:00,2026-08-31,https://nsearchives.nseindia.com/corporate/GIC...,NSE,106763376,https://nsearchives.nseindia.com/corporate/GIC...,True
7,9,3IINFOLTD,INE748C01038,3i Infotech Limited,Change in Director(s),3i Infotech Limited has informed the Exchange ...,Change in Director(s),historical_announcement,2026-08-31 23:23:09,2026-08-31,https://nsearchives.nseindia.com/corporate/3II...,NSE,106763371,https://nsearchives.nseindia.com/corporate/3II...,True
8,2509,WELSPUNLIV,INE192B01031,Welspun Living Limited,General Updates,Welspun Living Limited has informed the Exchan...,General Updates,historical_announcement,2026-08-31 23:21:49,2026-08-31,https://nsearchives.nseindia.com/corporate/WEL...,NSE,106763368,https://nsearchives.nseindia.com/corporate/WEL...,True
9,1561,NIITMTS,INE342G01023,NIIT Learning Systems Limited,Other Restructuring,"Update of the MST Holding GmbH Germany, a step...",Other Restructuring,historical_announcement,2026-08-31 23:17:19,2026-08-31,https://nsearchives.nseindia.com/corporate/NLS...,NSE,106763366,https://nsearchives.nseindia.com/corporate/NLS...,True


In [44]:
print("Null values:")
display(
    nse_insert_df.isna().sum()
)

print("\nDuplicate NSE IDs:")
print(
    nse_insert_df["nse_seq_id"].duplicated().sum()
)

print("\nUnique NSE IDs:")
print(
    nse_insert_df["nse_seq_id"].nunique()
)

Null values:


company_id        0
symbol            0
isin              0
company_name      0
headline          0
content           0
event_type        0
feed_type         0
published_at      0
event_date        0
url               0
source            0
nse_seq_id        0
attachment_url    0
hasXbrl           0
dtype: int64


Duplicate NSE IDs:
0

Unique NSE IDs:
20775


In [45]:
import mysql.connector
import os

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=False
)

cursor = conn.cursor()

insert_query = """
INSERT INTO news (
    company_id,
    symbol,
    isin,
    company_name,
    headline,
    content,
    event_type,
    feed_type,
    published_at,
    event_date,
    date_status,
    url,
    source,
    nse_seq_id,
    attachment_url,
    has_xbrl
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s
)
ON DUPLICATE KEY UPDATE
    content = VALUES(content),
    event_type = VALUES(event_type),
    published_at = VALUES(published_at),
    event_date = VALUES(event_date),
    date_status = VALUES(date_status),
    url = VALUES(url),
    attachment_url = VALUES(attachment_url),
    has_xbrl = VALUES(has_xbrl)
"""

rows = [
    tuple(row)
    for row in nse_insert_df[
        [
            "company_id",
            "symbol",
            "isin",
            "company_name",
            "headline",
            "content",
            "event_type",
            "feed_type",
            "published_at",
            "event_date",
            "url",
            "source",
            "nse_seq_id",
            "attachment_url",
            "hasXbrl"
        ]
    ].itertuples(index=False, name=None)
]

# Add date_status to every row
rows = [
    row[:10] + ("published",) + row[10:]
    for row in rows
]

cursor.executemany(insert_query, rows)

conn.commit()

print("Rows processed:", cursor.rowcount)

cursor.close()
conn.close()

Rows processed: 0


In [46]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT COUNT(*)
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

print("NSE announcements in DB:", cursor.fetchone()[0])

cursor.execute("""
    SELECT
        MIN(published_at),
        MAX(published_at),
        COUNT(DISTINCT company_id),
        COUNT(DISTINCT nse_seq_id)
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

print(cursor.fetchone())

cursor.close()
conn.close()

NSE announcements in DB: 20775
(datetime.datetime(2026, 8, 1, 8, 11, 24), datetime.datetime(2026, 8, 31, 23, 48, 18), 2265, 20775)


In [47]:
import mysql.connector
import os
import pandas as pd

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT company_id) AS companies,
        COUNT(DISTINCT symbol) AS symbols,
        COUNT(DISTINCT nse_seq_id) AS unique_nse_ids,
        MIN(published_at) AS oldest,
        MAX(published_at) AS newest
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

result = cursor.fetchone()

print("Total NSE news:", result[0])
print("Companies:", result[1])
print("Symbols:", result[2])
print("Unique NSE IDs:", result[3])
print("Oldest:", result[4])
print("Newest:", result[5])

cursor.close()
conn.close()

Total NSE news: 20775
Companies: 2265
Symbols: 2266
Unique NSE IDs: 20775
Oldest: 2026-08-01 08:11:24
Newest: 2026-08-31 23:48:18


In [48]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    id,
    company_id,
    symbol,
    company_name,
    headline,
    LEFT(content, 250) AS content_preview,
    event_type,
    published_at,
    nse_seq_id
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
ORDER BY published_at DESC
LIMIT 10
"""

sample_df = pd.read_sql(query, conn)

display(sample_df)

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/3522386781.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample_df = pd.read_sql(query, conn)


,id,company_id,symbol,company_name,headline,content_preview,event_type,published_at,nse_seq_id
0,265,1500,MUTHOOTFIN,Muthoot Finance Limited,Shareholders meeting,Muthoot Finance Limited has informed the Excha...,Shareholders meeting,2026-08-31 23:48:18,106763396
1,266,798,GICRE,General Insurance Corporation of India,Shareholders meeting,General Insurance Corporation of India has inf...,Shareholders meeting,2026-08-31 23:36:42,106763385
2,267,1158,JUSTDIAL,Just Dial Limited,Change in Director(s),Just Dial Limited has informed the Exchange re...,Change in Director(s),2026-08-31 23:36:14,106763384
3,268,1158,JUSTDIAL,Just Dial Limited,Shareholders meeting,Just Dial Limited has submitted the Exchange a...,Shareholders meeting,2026-08-31 23:31:38,106763382
4,269,640,E2E,E2E Networks Limited,Press Release,E2E Networks Limited has informed the Exchange...,Press Release,2026-08-31 23:27:56,106763381
5,270,2294,TI,Tilaknagar Industries Limited,Record Date,Tilaknagar Industries Limited has informed the...,Record Date,2026-08-31 23:26:24,106763379
6,271,798,GICRE,General Insurance Corporation of India,Copy of Newspaper Publication,General Insurance Corporation of India has inf...,Copy of Newspaper Publication,2026-08-31 23:24:00,106763376
7,272,9,3IINFOLTD,3i Infotech Limited,Change in Director(s),3i Infotech Limited has informed the Exchange ...,Change in Director(s),2026-08-31 23:23:09,106763371
8,273,2509,WELSPUNLIV,Welspun Living Limited,General Updates,Welspun Living Limited has informed the Exchan...,General Updates,2026-08-31 23:21:49,106763368
9,274,1561,NIITMTS,NIIT Learning Systems Limited,Other Restructuring,"Update of the MST Holding GmbH Germany, a step...",Other Restructuring,2026-08-31 23:17:19,106763366


In [49]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    event_type,
    COUNT(*) AS count
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
GROUP BY event_type
ORDER BY count DESC
"""

event_df = pd.read_sql(query, conn)

display(event_df.head(30))

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/2823220855.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  event_df = pd.read_sql(query, conn)


,event_type,count
0,Analysts/Institutional Investor Meet/Con. Call...,3442
1,Copy of Newspaper Publication,3043
2,General Updates,2185
3,Outcome of Board Meeting,2129
4,Shareholders meeting,2002
5,Updates,1394
6,Press Release,966
7,Investor Presentation,724
8,Record Date,498
9,Appointment,497


In [50]:
event_importance = {
    # HIGH IMPACT
    "Acquisition": "HIGH",
    "Bagging/Receiving of orders/contracts": "HIGH",
    "Credit Rating": "HIGH",
    "Change in Management": "HIGH",
    "Change in Auditors": "HIGH",
    "Outcome of Board Meeting": "HIGH",
    "Pendency of Litigation(s)/dispute(s) or the outcome": "HIGH",
    "Action(s) taken or orders passed": "HIGH",
    "Corporate Insolvency Resolution Process": "HIGH",
    "Allotment of Securities": "HIGH",
    "Disclosure under SEBI Takeover Regulations": "HIGH",
    "Disclosure of material issue": "HIGH",
    "Cessation": "HIGH",
    "Resignation": "HIGH",
    "Resignation of Director/KMP/SMP": "HIGH",

    # MEDIUM IMPACT
    "Appointment": "MEDIUM",
    "Change in Director(s)": "MEDIUM",
    "ESOP/ESOS/ESPS": "MEDIUM",
    "Monitoring Agency Report": "MEDIUM",
    "Statement of deviation(s) or variation(s) under": "MEDIUM",
    "Shareholders meeting": "MEDIUM",
    "Record Date": "MEDIUM",
    "Press Release": "MEDIUM",
    "Investor Presentation": "MEDIUM",
    "General Updates": "MEDIUM",
    "Updates": "MEDIUM",

    # LOW / ROUTINE
    "Copy of Newspaper Publication": "LOW",
    "Analysts/Institutional Investor Meet/Con. Call Updates": "LOW",
    "Trading Window": "LOW",
    "Price movement": "LOW",
    "Spurt in Volume": "LOW",
}

In [51]:
def classify_event_importance(event_type):
    if not isinstance(event_type, str):
        return "LOW"

    event = event_type.lower().strip()

    high_keywords = [
        "acquisition",
        "order",
        "contract",
        "credit rating",
        "change in management",
        "change in auditors",
        "outcome of board meeting",
        "litigation",
        "dispute",
        "insolvency",
        "allotment of securities",
        "takeover",
        "material issue",
        "cessation",
        "resignation",
    ]

    medium_keywords = [
        "appointment",
        "change in director",
        "esop",
        "monitoring agency",
        "deviation",
        "shareholders meeting",
        "record date",
        "press release",
        "investor presentation",
        "general updates",
        "updates",
    ]

    low_keywords = [
        "newspaper publication",
        "analysts",
        "institutional investor meet",
        "con. call",
        "trading window",
        "price movement",
        "spurt in volume",
    ]

    if any(keyword in event for keyword in high_keywords):
        return "HIGH"

    if any(keyword in event for keyword in medium_keywords):
        return "MEDIUM"

    if any(keyword in event for keyword in low_keywords):
        return "LOW"

    return "LOW"

In [52]:
event_df["importance"] = event_df["event_type"].apply(
    classify_event_importance
)

display(
    event_df.sort_values(
        ["importance", "count"],
        ascending=[True, False]
    )
)

,event_type,count,importance
3,Outcome of Board Meeting,2129,HIGH
10,Change in Management,336,HIGH
12,Resignation of Director/KMP/SMP,253,HIGH
15,Resignation,161,HIGH
16,Credit Rating,155,HIGH
...,...,...,...
20,Statement of deviation(s) or variation(s) unde...,125,MEDIUM
39,Committee Meeting Updates,38,MEDIUM
47,Monthly Business Updates,22,MEDIUM
61,Press Release (Revised),10,MEDIUM


In [53]:
print(
    event_df.groupby("importance")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance
MEDIUM    12550
HIGH       4133
LOW        4092
Name: count, dtype: int64


In [54]:
low_events = event_df[
    event_df["importance"] == "LOW"
].sort_values("count", ascending=False)

display(low_events)

,event_type,count,importance
1,Copy of Newspaper Publication,3043,LOW
22,Spurt in Volume,95,LOW
26,Trading Window,62,LOW
27,Price movement,61,LOW
30,Dividend,57,LOW
...,...,...,...
104,Redemption,1,LOW
103,One time settlement,1,LOW
102,Increase in Authorised Capital,1,LOW
101,Amendment(s),1,LOW


In [55]:
print("LOW event types:", len(low_events))
print("LOW announcements:", low_events["count"].sum())

LOW event types: 73
LOW announcements: 4092


In [56]:
low_events = event_df[
    event_df["importance"] == "LOW"
].sort_values("count", ascending=False)

display(
    low_events[
        ["event_type", "count"]
    ].to_string(index=False)
)

'                                                                                 event_type  count\n                                                              Copy of Newspaper Publication   3043\n                                                                            Spurt in Volume     95\n                                                                             Trading Window     62\n                                                                             Price movement     61\n                                                                                   Dividend     57\n                                                               Integrated Filing- Financial     56\n                                                                       Amendment to AOA/MOA     56\n                                                                                 Agreements     51\n                                                                          News Verification     40\

In [57]:
event_df["importance_v2"] = ...

In [58]:
high_events = {
    "Acquisition",
    "Amalgamation/Merger",
    "One time settlement",
    "Corporate Debt Restructuring",
    "Fraud/Default/Arrest",
    "Initiation of Forensic Audit",
    "Disruption of Operations",
    "Closure of operations",
    "Granting/withdrawal/surrender/cancellation/suspension of key licenses/ regulatory approvals",
    "Sale or disposal",
    "Diversification/Disinvestment",
    "Capacity addition",
    "Commencement of commercial production/operations",
    "Product launch",
    "Qualified Institutional Placement",
    "Preferential issue",
    "Issue of Securities",
    "Rights Issue",
    "Buyback",
    "Public Announcement - Buyback of Shares",
    "Public Announcement-Open Offer",
    "Offer for sale",
    "Giving guarantees/indemnity/ becoming a surety for third party",
    "Utilisation of Funds",
    "Reasons for Delayed/Non-submission of Financial Results",
    "Clarification - Financial Results",
    "Reply to Clarification- Financial results",
    "Suspension of Trading",
    "Voluntary Delisting",
    "Demerger",
    "Scheme of Arrangement",
    "Other Restructuring",
    "Rescission/termination(s)",
    "Strikes/Lockouts/Disturbances",
    "Adoption of new line(s) of business",
    "Arrangements for strategic, technical, manufacturing, or marketing tie up",
    "Memorandum of Understanding/Agreements",
    "Agreements",
    "Trading Plan under PIT"
}

medium_events = {
    "Dividend",
    "Integrated Filing- Financial",
    "Amendment to AOA/MOA",
    "Options to purchase securities",
    "Change in Company Secretary/Compliance Officer",
    "Retirement",
    "Demise",
    "Structural Digital Database",
    "Certificate under SEBI (Depositories and Participants) Regulations, 2018",
    "Registrar & Share Transfer Agent Update",
    "Forfeiture",
    "Stock split",
    "Bonus",
    "Conversion",
    "Closure of Buy Back",
    "Post Buyback Public Announcement",
    "Redemption",
    "Increase in Authorised Capital",
    "Address Change",
    "Name Change",
    "Name and Symbol Change",
    "Effect(s) on listed entity due to changed regulatory framework applicable",
    "Delay/default in the payment of fines/penalties/dues etc. to authority",
    "Amendment(s)",
    "Addendum",
    "Corrigendum",
    "Rumour Verification - Regulation 30(11)",
    "News Verification"
}

low_events = {
    "Copy of Newspaper Publication",
    "Spurt in Volume",
    "Trading Window",
    "Price movement",
    "Integrated Filing- Governance",
    "Others",
    "Loss of Share Certificates"
}

def classify_event_v2(event_type):
    if event_type in high_events:
        return "HIGH"
    elif event_type in medium_events:
        return "MEDIUM"
    elif event_type in low_events:
        return "LOW"
    else:
        return "REVIEW"

In [59]:
event_df["importance_v2"] = event_df["event_type"].apply(
    classify_event_v2
)

print(
    event_df.groupby("importance_v2")["count"]
    .sum()
    .sort_values(ascending=False)
)

print("\nEvents needing review:")
display(
    event_df[
        event_df["importance_v2"] == "REVIEW"
    ].sort_values("count", ascending=False)
)

importance_v2
REVIEW    16554
LOW        3270
HIGH        537
MEDIUM      414
Name: count, dtype: int64

Events needing review:


,event_type,count,importance,importance_v2
0,Analysts/Institutional Investor Meet/Con. Call...,3442,MEDIUM,REVIEW
2,General Updates,2185,MEDIUM,REVIEW
3,Outcome of Board Meeting,2129,HIGH,REVIEW
4,Shareholders meeting,2002,MEDIUM,REVIEW
5,Updates,1394,MEDIUM,REVIEW
6,Press Release,966,MEDIUM,REVIEW
7,Investor Presentation,724,MEDIUM,REVIEW
8,Record Date,498,MEDIUM,REVIEW
9,Appointment,497,MEDIUM,REVIEW
10,Change in Management,336,HIGH,REVIEW


In [60]:
event_df["event_type_clean"] = (
    event_df["event_type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [61]:
def classify_event_v2(event_type):
    event_type = str(event_type).strip()

    if event_type in high_events:
        return "HIGH"

    elif event_type in medium_events:
        return "MEDIUM"

    elif event_type in low_events:
        return "LOW"

    else:
        return "REVIEW"

In [62]:
event_df["importance_v2"] = event_df["event_type_clean"].apply(
    classify_event_v2
)

In [63]:
print(
    event_df.groupby("importance_v2")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance_v2
REVIEW    16552
LOW        3270
HIGH        537
MEDIUM      416
Name: count, dtype: int64


In [64]:
review_events = event_df[
    event_df["importance_v2"] == "REVIEW"
].sort_values("count", ascending=False)

display(
    review_events[
        ["event_type_clean", "count"]
    ]
)

,event_type_clean,count
0,Analysts/Institutional Investor Meet/Con. Call...,3442
2,General Updates,2185
3,Outcome of Board Meeting,2129
4,Shareholders meeting,2002
5,Updates,1394
6,Press Release,966
7,Investor Presentation,724
8,Record Date,498
9,Appointment,497
10,Change in Management,336


In [65]:
import re

def classify_event_v3(event_type):
    if not isinstance(event_type, str):
        return "REVIEW"

    event = re.sub(r"\s+", " ", event_type.strip().lower())

    # =========================
    # HIGH IMPORTANCE
    # =========================

    high_patterns = [
        # Business / strategic
        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",
        "other restructuring",
        "corporate debt restructuring",
        "one time settlement",
        "sale or disposal",
        "diversification/disinvestment",
        "adoption of new line",
        "strategic, technical, manufacturing",
        "memorandum of understanding",
        "mou",
        "agreement",
        "arrangement",

        # Orders / business growth
        "bagging/receiving of orders",
        "awarding of order",
        "order(s)/contract(s)",
        "capacity addition",
        "commencement of commercial production",
        "product launch",

        # Financial / capital
        "credit rating",
        "qualified institutional placement",
        "preferential issue",
        "issue of securities",
        "rights issue",
        "allotment of securities",
        "buyback",
        "open offer",
        "offer for sale",
        "utilisation of funds",
        "increase in authorised capital",

        # Management
        "change in management",
        "resignation of director",
        "resignation of statutory auditor",
        "resignation",
        "cessation",

        # Legal / regulatory
        "litigation",
        "dispute",
        "insolvency",
        "forensic audit",
        "fraud/default/arrest",
        "fraud",
        "default",
        "arrest",
        "key licenses",
        "regulatory approvals",
        "orders passed",
        "action(s) initiated",
        "action(s) taken",
        "fines/penalties/dues",
        "suspension of trading",
        "voluntary delisting",

        # Operational risk
        "disruption of operations",
        "closure of operations",
        "strikes/lockouts/disturbances",
    ]

    # =========================
    # LOW IMPORTANCE
    # =========================

    low_patterns = [
        "copy of newspaper publication",
        "spurt in volume",
        "trading window",
        "price movement",
        "loss of share certificates",
        "integrated filing- governance",
        "integrated filing - governance",
        "structural digital database",
    ]

    # =========================
    # MEDIUM IMPORTANCE
    # =========================

    medium_patterns = [
        "dividend",
        "integrated filing- financial",
        "integrated filing - financial",
        "amendment to aoa/moa",
        "options to purchase securities",
        "change in company secretary",
        "compliance officer",
        "retirement",
        "demise",
        "record date",
        "revised record date",
        "appointment",
        "change in director",
        "esop",
        "monitoring agency",
        "deviation",
        "shareholders meeting",
        "press release",
        "investor presentation",
        "general updates",
        "monthly business updates",
        "committee meeting updates",
        "name change",
        "address change",
        "name and symbol change",
        "certificate under sebi",
        "registrar & share transfer agent",
        "forfeiture",
        "stock split",
        "bonus",
        "conversion",
        "redemption",
        "closure of buy back",
        "post buyback public announcement",
        "rumour verification",
        "news verification",
        "corrigendum",
        "addendum",
        "amendment",
        "effect(s) on listed entity",
    ]

    # Check HIGH first
    if any(pattern in event for pattern in high_patterns):
        return "HIGH"

    # Then LOW
    if any(pattern in event for pattern in low_patterns):
        return "LOW"

    # Then MEDIUM
    if any(pattern in event for pattern in medium_patterns):
        return "MEDIUM"

    return "REVIEW"

In [66]:
event_df["importance_v3"] = event_df["event_type_clean"].apply(
    classify_event_v3
)

In [67]:
print(
    event_df.groupby("importance_v3")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance_v3
MEDIUM    8105
REVIEW    7303
LOW       3279
HIGH      2088
Name: count, dtype: int64


In [68]:
review_events = event_df[
    event_df["importance_v3"] == "REVIEW"
].sort_values("count", ascending=False)

print("REVIEW event types:", len(review_events))
print("REVIEW announcements:", review_events["count"].sum())

display(
    review_events[
        ["event_type_clean", "count"]
    ].to_string(index=False)
)

REVIEW event types: 13
REVIEW announcements: 7303


'                                              event_type_clean  count\n        Analysts/Institutional Investor Meet/Con. Call Updates   3442\n                                      Outcome of Board Meeting   2129\n                                                       Updates   1394\n                                            Change in Auditors    125\n                    Disclosure under SEBI Takeover Regulations     60\n                                  Disclosure of material issue     59\n                             Clarification - Financial Results     31\nGiving guarantees/indemnity/ becoming a surety for third party     20\n                     Reply to Clarification- Financial results     16\n       Reasons for Delayed/Non-submission of Financial Results     10\n                                        Trading Plan under PIT      8\n                                     Rescission/termination(s)      6\n                                                        Others      3'

In [69]:
high_patterns += [
    "outcome of board meeting",
    "change in auditors",
    "disclosure under sebi takeover regulations",
    "disclosure of material issue",
    "clarification - financial results",
    "reply to clarification- financial results",
    "reasons for delayed/non-submission of financial results",
    "giving guarantees/indemnity",
    "trading plan under pit",
    "rescission/termination"
]

medium_patterns += [
    "updates"
]

low_patterns += [
    "analysts/institutional investor meet/con. call updates",
    "others"
]

NameError: name 'high_patterns' is not defined

In [70]:
import re

def classify_event_v4(event_type):
    if not isinstance(event_type, str):
        return "REVIEW"

    event = re.sub(r"\s+", " ", event_type.strip().lower())

    # =========================
    # HIGH IMPORTANCE
    # =========================
    high_patterns = [
        # Board / management
        "outcome of board meeting",
        "change in management",
        "change in auditors",
        "resignation",
        "cessation",

        # Business / strategic
        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",
        "other restructuring",
        "corporate debt restructuring",
        "one time settlement",
        "sale or disposal",
        "diversification/disinvestment",
        "adoption of new line",
        "strategic, technical, manufacturing",
        "memorandum of understanding",
        "agreement",
        "arrangement",

        # Orders / operations
        "bagging/receiving of orders",
        "awarding of order",
        "order(s)/contract(s)",
        "capacity addition",
        "commencement of commercial production",
        "product launch",
        "disruption of operations",
        "closure of operations",
        "strikes/lockouts/disturbances",

        # Financial / capital
        "credit rating",
        "qualified institutional placement",
        "preferential issue",
        "issue of securities",
        "allotment of securities",
        "rights issue",
        "buyback",
        "open offer",
        "offer for sale",
        "utilisation of funds",
        "increase in authorised capital",

        # Legal / regulatory
        "litigation",
        "dispute",
        "insolvency",
        "forensic audit",
        "fraud/default/arrest",
        "fraud",
        "default",
        "arrest",
        "key licenses",
        "regulatory approvals",
        "orders passed",
        "action(s) initiated",
        "action(s) taken",
        "fines/penalties/dues",
        "suspension of trading",
        "voluntary delisting",

        # Financial-result issues
        "clarification - financial results",
        "reply to clarification- financial results",
        "reasons for delayed/non-submission of financial results",

        # SEBI / material disclosure
        "disclosure under sebi takeover regulations",
        "disclosure of material issue",

        # Guarantees / obligations
        "giving guarantees/indemnity",

        # Trading plan
        "trading plan under pit",

        # Termination
        "rescission/termination"
    ]

    # =========================
    # MEDIUM IMPORTANCE
    # =========================
    medium_patterns = [
        "dividend",
        "integrated filing- financial",
        "integrated filing - financial",
        "amendment to aoa/moa",
        "options to purchase securities",
        "change in company secretary",
        "compliance officer",
        "retirement",
        "demise",
        "record date",
        "revised record date",
        "appointment",
        "change in director",
        "esop",
        "monitoring agency",
        "deviation",
        "shareholders meeting",
        "press release",
        "investor presentation",
        "general updates",
        "updates",
        "monthly business updates",
        "committee meeting updates",
        "name change",
        "address change",
        "name and symbol change",
        "certificate under sebi",
        "registrar & share transfer agent",
        "forfeiture",
        "stock split",
        "bonus",
        "conversion",
        "redemption",
        "closure of buy back",
        "post buyback public announcement",
        "rumour verification",
        "news verification",
        "corrigendum",
        "addendum",
        "amendment",
        "effect(s) on listed entity"
    ]

    # =========================
    # LOW IMPORTANCE
    # =========================
    low_patterns = [
        "copy of newspaper publication",
        "spurt in volume",
        "trading window",
        "price movement",
        "loss of share certificates",
        "integrated filing- governance",
        "integrated filing - governance",
        "structural digital database",
        "analysts/institutional investor meet/con. call updates",
        "others"
    ]

    # HIGH first
    if any(pattern in event for pattern in high_patterns):
        return "HIGH"

    # LOW second
    if any(pattern in event for pattern in low_patterns):
        return "LOW"

    # MEDIUM third
    if any(pattern in event for pattern in medium_patterns):
        return "MEDIUM"

    return "REVIEW"

In [71]:
event_df["importance_v4"] = event_df["event_type_clean"].apply(
    classify_event_v4
)

In [72]:
print(
    event_df.groupby("importance_v4")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance_v4
MEDIUM    9508
LOW       6724
HIGH      4543
Name: count, dtype: int64


In [73]:
review_events = event_df[
    event_df["importance_v4"] == "REVIEW"
].sort_values("count", ascending=False)

print("REVIEW event types:", len(review_events))
print("REVIEW announcements:", review_events["count"].sum())

display(
    review_events[
        ["event_type_clean", "count"]
    ]
)

REVIEW event types: 0
REVIEW announcements: 0


,event_type_clean,count


In [74]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("SHOW COLUMNS FROM news LIKE 'importance'")

result = cursor.fetchone()

print(result)

cursor.close()
conn.close()

('importance', "enum('HIGH','MEDIUM','LOW')", 'YES', '', None, '')


In [76]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
    ADD COLUMN importance ENUM('HIGH', 'MEDIUM', 'LOW') NULL
    AFTER event_type
""")

conn.commit()

print("importance column added")

cursor.close()
conn.close()

ProgrammingError: 1060 (42S21): Duplicate column name 'importance'

In [77]:
event_df["importance_v4"]

0         LOW
1         LOW
2      MEDIUM
3        HIGH
4      MEDIUM
        ...  
105      HIGH
106    MEDIUM
107       LOW
108      HIGH
109    MEDIUM
Name: importance_v4, Length: 110, dtype: object

In [78]:
importance_map = (
    event_df
    .set_index("event_type_clean")["importance_v4"]
    .to_dict()
)

investiq_news["event_type_clean"] = (
    investiq_news["event_type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

investiq_news["importance"] = (
    investiq_news["event_type_clean"]
    .map(importance_map)
)

print(
    investiq_news["importance"].value_counts(dropna=False)
)

importance
MEDIUM    9508
LOW       6724
HIGH      4543
Name: count, dtype: int64


In [79]:
print(
    "Missing importance:",
    investiq_news["importance"].isna().sum()
)

Missing importance: 0


In [80]:
import mysql.connector
import os

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=False
)

cursor = conn.cursor()

update_query = """
UPDATE news
SET importance = %s
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
  AND nse_seq_id = %s
"""

rows = [
    (row.importance, int(row.nse_seq_id))
    for row in investiq_news[
        ["importance", "nse_seq_id"]
    ].itertuples(index=False)
]

cursor.executemany(update_query, rows)

conn.commit()

print("Rows updated:", cursor.rowcount)

cursor.close()
conn.close()

Rows updated: 20775


In [81]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    importance,
    COUNT(*) AS count
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
GROUP BY importance
ORDER BY
    FIELD(importance, 'HIGH', 'MEDIUM', 'LOW')
"""

db_importance_df = pd.read_sql(query, conn)

display(db_importance_df)

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/231563556.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_importance_df = pd.read_sql(query, conn)


,importance,count
0,HIGH,4543
1,MEDIUM,9508
2,LOW,6724


In [82]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(importance = 'HIGH') AS high_count,
        SUM(importance = 'MEDIUM') AS medium_count,
        SUM(importance = 'LOW') AS low_count,
        SUM(importance IS NULL) AS null_count
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

print(cursor.fetchone())

cursor.close()
conn.close()

(20775, Decimal('4543'), Decimal('9508'), Decimal('6724'), Decimal('0'))


In [83]:
import mysql.connector
import os
import pandas as pd

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    id,
    company_id,
    symbol,
    isin,
    company_name,
    headline,
    content,
    event_type,
    feed_type,
    published_at,
    url,
    source,
    nse_seq_id
FROM news
WHERE source = 'NSE'
  AND feed_type != 'historical_announcement'
ORDER BY published_at DESC
"""

rss_db_df = pd.read_sql(query, conn)

conn.close()

print("RSS records in DB:", len(rss_db_df))
display(rss_db_df.head(10))

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/4090382548.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rss_db_df = pd.read_sql(query, conn)


RSS records in DB: 66


,id,company_id,symbol,isin,company_name,headline,content,event_type,feed_type,published_at,url,source,nse_seq_id
0,199,666,ELGIEQUIP,INE285A01027,Elgi Equipments Limited,Elgi Equipments Limited has informed the Excha...,None,None,online_announcement,2026-08-29 07:48:32,https://nsearchives.nseindia.com/corporate/ELG...,NSE,None
1,200,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,SIGNPOST INDIA LIMITED has informed the Exchan...,None,None,online_announcement,2026-08-29 06:32:55,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
2,201,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchan...,None,None,online_announcement,2026-08-29 05:29:08,https://nsearchives.nseindia.com/corporate/SIG...,NSE,None
3,202,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchan...,None,None,online_announcement,2026-08-29 05:25:44,https://nsearchives.nseindia.com/corporate/SIG...,NSE,None
4,203,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchan...,None,None,online_announcement,2026-08-29 05:23:00,https://nsearchives.nseindia.com/corporate/SIG...,NSE,None
5,204,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 02:04:10,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
6,205,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:47:37,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
7,206,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:47:29,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
8,207,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:39:08,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
9,208,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:38:59,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None


In [84]:
print(
    rss_db_df.groupby(["source", "feed_type"])
    .size()
    .sort_values(ascending=False)
)

source  feed_type                
NSE     online_announcement          28
        annual_report                20
        related_party_transaction    14
        financial_result              4
dtype: int64


In [85]:
print(
    "RSS records with NSE ID:",
    rss_db_df["nse_seq_id"].notna().sum()
)

print(
    "RSS records without NSE ID:",
    rss_db_df["nse_seq_id"].isna().sum()
)

RSS records with NSE ID: 0
RSS records without NSE ID: 66


In [86]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

nse_query = """
SELECT
    id,
    company_id,
    symbol,
    company_name,
    headline,
    content,
    event_type,
    published_at,
    event_date,
    url,
    nse_seq_id
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
"""

nse_db_df = pd.read_sql(nse_query, conn)

conn.close()

print("NSE historical records:", len(nse_db_df))

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/3735595186.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  nse_db_df = pd.read_sql(nse_query, conn)


NSE historical records: 20775


In [87]:
import re

def normalize_news_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [88]:
rss_db_df["text_norm"] = (
    rss_db_df["headline"].fillna("") + " " +
    rss_db_df["content"].fillna("")
).apply(normalize_news_text)

nse_db_df["text_norm"] = (
    nse_db_df["headline"].fillna("") + " " +
    nse_db_df["content"].fillna("")
).apply(normalize_news_text)

In [89]:
nse_text_lookup = {}

for idx, row in nse_db_df.iterrows():
    text = row["text_norm"]

    if text:
        nse_text_lookup.setdefault(text, []).append(idx)

rss_db_df["exact_text_match"] = rss_db_df["text_norm"].apply(
    lambda x: nse_text_lookup.get(x, [])
)

print(
    "RSS records with exact text match:",
    (rss_db_df["exact_text_match"].str.len() > 0).sum()
)

RSS records with exact text match: 0


In [90]:
nse_url_set = set(
    nse_db_df["url"]
    .dropna()
    .astype(str)
    .str.strip()
)

rss_db_df["url_match"] = rss_db_df["url"].astype(str).str.strip().isin(
    nse_url_set
)

print(
    "RSS records with matching URL:",
    rss_db_df["url_match"].sum()
)

RSS records with matching URL: 14


In [91]:
from rapidfuzz.fuzz import ratio

# Create a company-based lookup to avoid comparing
# every RSS record against all 20,775 NSE records.

nse_by_company = {
    company_id: group.copy()
    for company_id, group in nse_db_df.groupby("company_id")
}

matches = []

for _, rss in rss_db_df.iterrows():

    company_records = nse_by_company.get(rss["company_id"])

    if company_records is None:
        matches.append({
            "rss_id": rss["id"],
            "best_nse_id": None,
            "similarity": 0
        })
        continue

    rss_text = normalize_news_text(rss["headline"])

    best_score = 0
    best_nse_id = None

    for _, nse in company_records.iterrows():

        nse_text = normalize_news_text(nse["content"])

        if not rss_text or not nse_text:
            continue

        score = ratio(rss_text, nse_text)

        if score > best_score:
            best_score = score
            best_nse_id = nse["id"]

    matches.append({
        "rss_id": rss["id"],
        "best_nse_id": best_nse_id,
        "similarity": best_score
    })

rss_match_df = pd.DataFrame(matches)

display(
    rss_match_df.sort_values(
        "similarity",
        ascending=False
    ).head(20)
)

,rss_id,best_nse_id,similarity
4,203,1336.0,96.279070
2,201,1334.0,96.172249
22,221,1341.0,91.292876
21,220,1340.0,91.129032
25,224,1344.0,90.651558
3,202,1335.0,89.454545
24,223,1343.0,88.796680
19,218,1338.0,88.259109
27,226,1346.0,86.192469
26,225,1346.0,86.192469


In [92]:
rss_match_df = (
    rss_match_df
    .merge(
        rss_db_df[
            ["id", "symbol", "company_name", "published_at", "headline"]
        ],
        left_on="rss_id",
        right_on="id",
        how="left",
        suffixes=("", "_rss")
    )
    .merge(
        nse_db_df[
            ["id", "published_at", "event_type", "content", "nse_seq_id"]
        ],
        left_on="best_nse_id",
        right_on="id",
        how="left",
        suffixes=("", "_nse")
    )
)

display(
    rss_match_df[
        [
            "rss_id",
            "symbol",
            "published_at",
            "best_nse_id",
            "published_at_nse",
            "similarity",
            "event_type",
            "nse_seq_id"
        ]
    ].sort_values("similarity", ascending=False)
)

,rss_id,symbol,published_at,best_nse_id,published_at_nse,similarity,event_type,nse_seq_id
4,203,SIGNPOST,2026-08-29 05:23:00,1336.0,2026-08-29 05:23:00,96.279070,Updates,106761105.0
2,201,SIGNPOST,2026-08-29 05:29:08,1334.0,2026-08-29 05:29:08,96.172249,Updates,106761107.0
22,221,MAXESTATES,2026-08-29 00:19:46,1341.0,2026-08-29 00:19:46,91.292876,Outcome of Board Meeting,106761059.0
21,220,MAXESTATES,2026-08-29 00:22:03,1340.0,2026-08-29 00:22:03,91.129032,Press Release,106761060.0
25,224,MAXESTATES,2026-08-29 00:08:36,1344.0,2026-08-29 00:08:36,90.651558,Outcome of Board Meeting,106761056.0
...,...,...,...,...,...,...,...,...
45,264,SFML,2025-02-21 15:12:33,NaN,NaT,0.000000,NaN,NaN
44,263,LLOYDS,2025-04-25 21:12:12,NaN,NaT,0.000000,NaN,NaN
43,262,GANESHIN,2025-04-26 19:14:11,NaN,NaT,0.000000,NaN,NaN
42,261,QUESTLAB,2025-04-28 14:07:07,NaN,NaT,0.000000,NaN,NaN


In [93]:
# Add exact time difference between RSS and best NSE match

rss_match_df["published_at"] = pd.to_datetime(
    rss_match_df["published_at"],
    errors="coerce"
)

rss_match_df["published_at_nse"] = pd.to_datetime(
    rss_match_df["published_at_nse"],
    errors="coerce"
)

rss_match_df["time_diff_minutes"] = (
    (rss_match_df["published_at"] - rss_match_df["published_at_nse"])
    .abs()
    .dt.total_seconds()
    / 60
)

# Candidate duplicate rule
rss_match_df["duplicate_candidate"] = (
    (rss_match_df["best_nse_id"].notna()) &
    (rss_match_df["similarity"] >= 80) &
    (rss_match_df["time_diff_minutes"] <= 10)
)

print(
    "Duplicate candidates:",
    rss_match_df["duplicate_candidate"].sum()
)

display(
    rss_match_df[
        [
            "rss_id",
            "symbol",
            "published_at",
            "best_nse_id",
            "published_at_nse",
            "time_diff_minutes",
            "similarity",
            "event_type",
            "nse_seq_id",
            "duplicate_candidate"
        ]
    ]
    .sort_values(
        ["duplicate_candidate", "similarity"],
        ascending=[False, False]
    )
)

Duplicate candidates: 14


,rss_id,symbol,published_at,best_nse_id,published_at_nse,time_diff_minutes,similarity,event_type,nse_seq_id,duplicate_candidate
4,203,SIGNPOST,2026-08-29 05:23:00,1336.0,2026-08-29 05:23:00,0.0,96.279070,Updates,106761105.0,True
2,201,SIGNPOST,2026-08-29 05:29:08,1334.0,2026-08-29 05:29:08,0.0,96.172249,Updates,106761107.0,True
22,221,MAXESTATES,2026-08-29 00:19:46,1341.0,2026-08-29 00:19:46,0.0,91.292876,Outcome of Board Meeting,106761059.0,True
21,220,MAXESTATES,2026-08-29 00:22:03,1340.0,2026-08-29 00:22:03,0.0,91.129032,Press Release,106761060.0,True
25,224,MAXESTATES,2026-08-29 00:08:36,1344.0,2026-08-29 00:08:36,0.0,90.651558,Outcome of Board Meeting,106761056.0,True
...,...,...,...,...,...,...,...,...,...,...
51,250,AHIMSA,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False
52,243,PCCL,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False
58,237,SKP,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False
59,236,OMFURN,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False


In [94]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
    ADD COLUMN is_duplicate TINYINT(1) NOT NULL DEFAULT 0
    AFTER importance
""")

cursor.execute("""
    ALTER TABLE news
    ADD COLUMN duplicate_of BIGINT NULL
    AFTER is_duplicate
""")

conn.commit()

print("Duplicate tracking columns added.")

cursor.close()
conn.close()

Duplicate tracking columns added.


In [95]:
duplicate_matches = rss_match_df[
    rss_match_df["duplicate_candidate"] == True
][
    ["rss_id", "best_nse_id"]
].copy()

print("Duplicates to mark:", len(duplicate_matches))

display(duplicate_matches)

Duplicates to mark: 14


,rss_id,best_nse_id
0,199,1333.0
2,201,1334.0
3,202,1335.0
4,203,1336.0
18,217,1337.0
19,218,1338.0
20,219,1339.0
21,220,1340.0
22,221,1341.0
23,222,1342.0


In [96]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=False
)

cursor = conn.cursor()

update_query = """
UPDATE news
SET
    is_duplicate = 1,
    duplicate_of = %s
WHERE id = %s
"""

rows = [
    (int(row.best_nse_id), int(row.rss_id))
    for row in duplicate_matches.itertuples(index=False)
]

cursor.executemany(update_query, rows)

conn.commit()

print("RSS duplicates marked:", cursor.rowcount)

cursor.close()
conn.close()

RSS duplicates marked: 14


In [97]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        is_duplicate,
        COUNT(*)
    FROM news
    GROUP BY is_duplicate
    ORDER BY is_duplicate
""")

print(cursor.fetchall())

cursor.execute("""
    SELECT
        COUNT(*)
    FROM news
    WHERE is_duplicate = 1
      AND duplicate_of IS NOT NULL
""")

print("Linked duplicates:", cursor.fetchone()[0])

cursor.close()
conn.close()

[(0, 20827), (1, 14)]
Linked duplicates: 14


In [98]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    source,
    feed_type,
    COUNT(*) AS count
FROM news
GROUP BY source, feed_type
ORDER BY source, feed_type
"""

df = pd.read_sql(query, conn)

display(df)

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/2114766429.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,source,feed_type,count
0,NSE,annual_report,20
1,NSE,financial_result,4
2,NSE,historical_announcement,20775
3,NSE,online_announcement,28
4,NSE,related_party_transaction,14


In [99]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(is_duplicate = 1) AS duplicates,
        SUM(is_duplicate = 0) AS originals
    FROM news
""")

print(cursor.fetchone())

cursor.close()
conn.close()

(20841, Decimal('14'), Decimal('20827'))


In [100]:
display(
    rss_match_df[
        rss_match_df["best_nse_id"] == 1346
    ][
        [
            "rss_id",
            "symbol",
            "published_at",
            "best_nse_id",
            "published_at_nse",
            "similarity",
            "event_type",
            "nse_seq_id"
        ]
    ]
)

,rss_id,symbol,published_at,best_nse_id,published_at_nse,similarity,event_type,nse_seq_id
5,204,MAXESTATES,2026-08-29 02:04:10,1346.0,2026-08-29 00:02:45,67.811159,Outcome of Board Meeting,106761054.0
6,205,MAXESTATES,2026-08-29 01:47:37,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
7,206,MAXESTATES,2026-08-29 01:47:29,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
8,207,MAXESTATES,2026-08-29 01:39:08,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
9,208,MAXESTATES,2026-08-29 01:38:59,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
10,209,MAXESTATES,2026-08-29 01:38:47,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
11,210,MAXESTATES,2026-08-29 01:22:55,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
12,211,MAXESTATES,2026-08-29 01:22:43,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
13,212,MAXESTATES,2026-08-29 01:22:25,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
14,213,MAXESTATES,2026-08-29 01:22:05,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
